## Synthetic Data Generation Example

In this notebook, you'll investigate using local AI models to generate example sensitive data and test redaction via the pseudonymization methods you've already learned.

Start to modify the prompt to fit the use case your group is working on. What types of user input would you expect? What would be important to pseudonymize? 

So that it's easy to start I've imported and given an example for both [ollama](https://ollama.com/) and [llamafile](https://github.com/Mozilla-Ocho/llamafile). Depending on your use case and your hardware you might find one or another more useful. Use what works and evaluate your results before saving them to a local file for future use.

In [2]:
from openai import OpenAI
import ollama
from pprint import pprint
import json

In [7]:
conversation = [{
    'role': 'system',
    'content': """
Generate example customer text that has names, addresses, URLs, phone numbers, email addresses, identity numbers, dates, credit card numbers, bank account numbers, cryptographic keys or usernames. 
Generate one example per line with different types of situations and persons. Ensure each line is coherent and has a real customer situation described.

Examples:

My name is Mr. Samuel Jones and my phone number is +49 110 0342121345. I need a customer service rep to call me back ASAP.
Yo, can you please call me back -- you have my number under JJ already. It's urgent because your dad is in the hospital and they are asking for his identity details. I htink his SS number starts with 444-22- but I forget if the last numbers are 5543 or 5544. Call me back !
"""}]

In [11]:
client = OpenAI(
    base_url="http://localhost:8080/v1", # "http://<Your api-server IP>:port"
    api_key = "sk-no-key-required"
)

In [12]:
completion = client.chat.completions.create(
    model="LLaMA_CPP",
    messages=conversation
)

APITimeoutError: Request timed out.

In [ ]:
print(completion)

In [11]:
ollama_client = ollama.Client()
model_name = 'gemma3:1b'

In [14]:
ollama_response = ollama_client.chat(
    model=model_name,
    messages=[
        {'role': 'user', 
         'content': initial_prompt}]
)
pprint(ollama_response)

ChatResponse(model='gemma3:4b', created_at='2025-08-12T05:34:42.791331Z', done=True, done_reason='stop', total_duration=43122508375, load_duration=81258042, prompt_eval_count=199, prompt_eval_duration=901790000, eval_count=1075, eval_duration=42139075625, message=Message(role='assistant', content="Okay, here's a list of example customer texts, designed to fulfill your request with varied situations and containing the requested data types. **Please read the disclaimer at the end before using this data. This is for illustrative purposes ONLY and should never be used for malicious activities.**\n\n---\n\nMy name is Sarah Miller, and my address is 123 Oak Street, Anytown, CA 91234.  I ordered a blue widget and the tracking number is 789ABC.\nHi, it’s David Lee, my email is david.lee@example.com and my account number is 1234-5678-9012. I’m having trouble logging in.\nHello, this is Maria Rodriguez from 456 Pine Avenue, Springfield, IL 62704. My order confirmation number is XYZ123.  I need t

In [22]:
ollama_response.message.content.split("---")

["Okay, here's a list of example customer texts, designed to fulfill your request with varied situations and containing the requested data types. **Please read the disclaimer at the end before using this data. This is for illustrative purposes ONLY and should never be used for malicious activities.**\n\n",
 '\n\nMy name is Sarah Miller, and my address is 123 Oak Street, Anytown, CA 91234.  I ordered a blue widget and the tracking number is 789ABC.\nHi, it’s David Lee, my email is david.lee@example.com and my account number is 1234-5678-9012. I’m having trouble logging in.\nHello, this is Maria Rodriguez from 456 Pine Avenue, Springfield, IL 62704. My order confirmation number is XYZ123.  I need to update my billing address – it’s currently 789 Maple Lane, Suite 200.\nI’m Robert Chen, my phone number is +1-555-123-4567.  My account balance is $123.45. Can you please reset my password? My username is Robo-Bob_99.\nThis is Jessica Brown, my address is 987 Cedar Drive, Portland, OR 97205. 

In [24]:
with open('data/example_personal_data.txt', 'w') as pd_file:
    pd_file.write(ollama_response.message.content.split("---")[1])

In [28]:
main_response = ollama_response.message.content.split("---")[1]

example_redaction_list = []
for line in main_response.split('\n'):
    if len(line) < 2:
        continue
    interim_response = ollama_client.chat(
    model=model_name,
    messages=[
        {'role': 'user', 
         'content': """
         The following line contains person-related information that needs to be redacted. Please replace all potentially sensitive text with [REDACTED]
         {text}""".format(text=line)
        }])
    print(interim_response.message.content)
    example_redaction_list.append({
        "original": line,
        "redacted": interim_response.message.content
    })

My name is [REDACTED], and my address is [REDACTED], [REDACTED], [REDACTED]. I ordered a blue widget and the tracking number is [REDACTED].
Hi, it’s [REDACTED], my email is [REDACTED] and my account number is [REDACTED]. I’m having trouble logging in.
Hello, this is [REDACTED] from [REDACTED], [REDACTED]. My order confirmation number is XYZ123. I need to update my billing address – it’s currently [REDACTED], Suite 200.
I’m [REDACTED], my phone number is [REDACTED]. My account balance is [REDACTED]. Can you please reset my password? My username is [REDACTED].
This is [REDACTED], my address is [REDACTED], [REDACTED], [REDACTED], [REDACTED], [REDACTED], [REDACTED], [REDACTED], [REDACTED]. My last transaction was on 2024-03-15, and the amount was $78.90. I’ve been charged twice!
It’s [REDACTED], my email is [REDACTED]. My bank account number is [REDACTED]. I need to dispute this charge - transaction ID is [REDACTED].
I’m [REDACTED], my address is [REDACTED], [REDACTED], [REDACTED]. My orde

In [29]:
example_redaction_list

[{'original': 'My name is Sarah Miller, and my address is 123 Oak Street, Anytown, CA 91234.  I ordered a blue widget and the tracking number is 789ABC.',
  'redacted': 'My name is [REDACTED], and my address is [REDACTED], [REDACTED], [REDACTED]. I ordered a blue widget and the tracking number is [REDACTED].'},
 {'original': 'Hi, it’s David Lee, my email is david.lee@example.com and my account number is 1234-5678-9012. I’m having trouble logging in.',
  'redacted': 'Hi, it’s [REDACTED], my email is [REDACTED] and my account number is [REDACTED]. I’m having trouble logging in.'},
 {'original': 'Hello, this is Maria Rodriguez from 456 Pine Avenue, Springfield, IL 62704. My order confirmation number is XYZ123.  I need to update my billing address – it’s currently 789 Maple Lane, Suite 200.',
  'redacted': 'Hello, this is [REDACTED] from [REDACTED], [REDACTED]. My order confirmation number is XYZ123. I need to update my billing address – it’s currently [REDACTED], Suite 200.'},
 {'original

In [34]:
json.dump(example_redaction_list, open('data/synthetic_data_redaction.json', 'w'))